# 基于 VGG16 的图形分类系统

深度学习课程项目 · 图像分类 · PyTorch 复现

## 项目简介

使用 VGG16 卷积神经网络对 4 类几何图形（圆形 / 方形 / 三角形 / 星形）进行分类。
数据集由程序自动生成（带随机位置、大小、颜色、背景的合成图形），无需外部下载。

## 环境

```
python >= 3.8
torch, torchvision, pillow, numpy
```

安装：`pip install torch torchvision pillow numpy`

## 运行

直接顺序执行下面的代码格即可（CPU 约 20~40 分钟，GPU 数分钟）。

## 网络结构

- 主干：VGG16（13 个卷积层 + 3 个全连接层）
- 输入：64×64×3
- 输出：4 类
- 损失：交叉熵；优化器：Adam(lr=1e-3)


In [ ]:
import os, math, time, random, json
import numpy as np
from PIL import Image, ImageDraw
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

CLASSES = ["circle", "square", "triangle", "star"]
IMG_SIZE = 64
N_TRAIN_PER_CLASS = 800
N_TEST_PER_CLASS = 200
EPOCHS = 8
BATCH_SIZE = 64
LR = 1e-3

## 1. 数据集生成

`draw_shape` 随机绘制一个带随机位置 / 大小 / 颜色 / 背景色的图形，
训练集与测试集使用不同随机种子，保证不重叠。

In [ ]:
def draw_shape(kind, size=IMG_SIZE, rng=None):
    rng = rng or random
    bg = tuple(rng.randint(200, 255) for _ in range(3))
    img = Image.new("RGB", (size, size), bg)
    d = ImageDraw.Draw(img)
    m = rng.randint(6, 16)
    box = [m, m, size - m, size - m]
    color = tuple(rng.randint(0, 190) for _ in range(3))
    if kind == "circle":
        d.ellipse(box, fill=color)
    elif kind == "square":
        w = rng.randint(m, size - m)
        x0 = rng.randint(2, size - w - 2); y0 = rng.randint(2, size - w - 2)
        d.rectangle([x0, y0, x0 + w, y0 + w], fill=color)
    elif kind == "triangle":
        d.polygon([(size // 2, m), (m, size - m), (size - m, size - m)], fill=color)
    elif kind == "star":
        cx = cy = size // 2
        r = size // 2 - m
        pts = []
        for i in range(10):
            ang = -math.pi / 2 + i * math.pi / 5
            rr = r if i % 2 == 0 else r * 0.45
            pts.append((cx + rr * math.cos(ang), cy + rr * math.sin(ang)))
        d.polygon(pts, fill=color)
    return img


class ShapeDataset(Dataset):
    def __init__(self, n_per_class, seed):
        self.items = []
        rng = random.Random(seed)
        for ci, name in enumerate(CLASSES):
            for _ in range(n_per_class):
                self.items.append((draw_shape(name, rng=rng), ci))
        rng.shuffle(self.items)
        self.tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img, y = self.items[idx]
        return self.tf(img), y


train_ds = ShapeDataset(N_TRAIN_PER_CLASS, seed=1)
test_ds = ShapeDataset(N_TEST_PER_CLASS, seed=9999)
nw = 0 if os.name == "nt" else 2   # Windows 下必须用 0，否则多进程报错
train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=nw)
test_ld = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=nw)
print(f"训练集 {len(train_ds)} 张 / 测试集 {len(test_ds)} 张 / {len(CLASSES)} 类")

## 2. 模型定义

使用 torchvision 的 VGG16，把最后的分类头改为 4 类输出。

In [ ]:
model = models.vgg16(weights=None)
model.classifier[6] = nn.Linear(4096, len(CLASSES))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
model

## 3. 训练

In [ ]:
def evaluate(m, loader):
    m.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = m(x).argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


best_acc = 0.0
history = []
for ep in range(EPOCHS):
    model.train()
    loss_sum = correct = total = 0
    for x, y in train_ld:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += x.size(0)
    acc = evaluate(model, test_ld)
    best_acc = max(best_acc, acc)
    history.append({"epoch": ep + 1, "train_loss": round(loss_sum / total, 4),
                    "train_acc": round(correct / total, 4), "test_acc": round(acc, 4)})
    print(f"[epoch {ep + 1}] train_loss={loss_sum / total:.4f} "
          f"train_acc={correct / total:.4f}  test_acc={acc:.4f}")

print(f"\n>>> 真实测试集准确率: {best_acc * 100:.2f}%")

## 4. 推理耗时测试

In [ ]:
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    for _ in range(10):
        model(dummy)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(50):
        model(dummy)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    ms = (time.time() - t0) / 50 * 1000
print(f">>> 单张推理平均耗时: {ms:.1f} ms （{DEVICE}，batch=1）")

## 5. 保存模型与结果

In [ ]:
torch.save(model.state_dict(), "vgg16_shapes.pth")
with open("results.json", "w", encoding="utf-8") as f:
    json.dump({"classes": CLASSES, "img_size": IMG_SIZE, "epochs": EPOCHS,
               "best_test_acc": round(best_acc * 100, 2),
               "single_infer_ms": round(ms, 1),
               "history": history}, f, ensure_ascii=False, indent=2)
print("已保存 vgg16_shapes.pth 和 results.json")

## 实验结果

见下方 `results.json`。

## 后续优化

- 换用真实数据集（CIFAR-10 / TrashNet）
- 迁移学习（`weights="DEFAULT"`）快速提升精度
- 增加数据增强、学习率调度、模型量化部署